This notebook is used to investigate latent space. Specifically, it allows to apply different clustering/embedding methods, such as **t-SNE**, **k-means**, and **UMAP**. It plots random events of the defined region of a cluster.

In [ ]:
import os
import sys
import numpy as np
import random
sys.path.append('../../')
from simulated_data.plot_data import plot_event
from clustering import t_SNE_clustering
from clustering import UMAP_embedding
from clustering import k_means_clustering
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
# create a folder for results of global feature exploration
folder_path = f'../plots/exploration'
os.makedirs(folder_path, exist_ok=True)

Global features are generated using ***global_features_extraction.py***

In [ ]:
# load the global features
features = np.load('../global_features/O16_experimental_features.npy')

In [ ]:
# load labels
labels = np.load('../O16_Experimental_Labels.npy')

In [ ]:
# sort features in label_data for track identification(for O16 experimental labeled data)
#        0 is 0-,1-,2-track data
#        1 is 3-track data
#        2 is 4-,5-track data

indices = labels[:, 0].astype(int)
labels = labels[:, 1].astype(int)

labels = np.where(np.isin(labels, [0, 1, 2]), 0, labels)
labels = np.where(np.isin(labels, [3]), 1, labels)
labels = np.where(np.isin(labels, [4, 5]), 2, labels)

features = features[indices]

np.save('../global_features/O16_Exp_extr.npy', features)
np.save('../global_features/O16_Exp_labels', labels)

In [ ]:
# Separate features OR Comment if you have separated features
features_0 = features[labels == 0]
features_1 = features[labels == 1]
features_2 = features[labels == 2]

Applies **k-means** clustering from ***clusering.py***. Edit ***save_dir*** to specify where results will be stored. 

***Sample Invocation of k-means***:

    k_means_clustering(features, labels, dimension, save_dir, num_samples_to_print=10)

In [ ]:
save_dir='../plots/k_means'
label_names = ["0,1,2 Tracks", "3 Tracks", "4, 5 Tracks"]
features_glob, cluster_labels, cluster_indices = k_means_clustering(features[:, [31, 525]], labels, 2, save_dir, label_names, num_samples_to_print=10)

***Optional part for O16 Experimental Labeled Data***

Transforms indices sampled after clustering into ones that can be used to navigate through original events.

In [ ]:
def transform_ids(cluster_ids, original_ids):
    original = []

    for index in cluster_ids:
        original.append(original_ids[index])
    return original

clusters = []
print("Transformed indices:")
for i, cluster in enumerate(cluster_indices):
    clusters.append(transform_ids(cluster, indices))
    print(f"Cluster {i}: {clusters[i]}")

Prints true labels (Optional part for visualization).

In [ ]:
def extract_true_labels(indices, labels):
    c_labels = []
    for index in indices:
        c_labels.append(labels[index])

    return c_labels

print("True labels: ")
true_labels = []
for i, cluster in enumerate(cluster_indices):
    true_labels.append(extract_true_labels(cluster, labels))
    print(f"Cluster {i}: {true_labels[i]}")

Plots events using provided indices. Edit ***events*** to specify the location of the original data. 

In [ ]:
def plot_events(indices, name, events, true_labels, cols=3):
    rows = int(np.ceil(len(indices) / cols))
    fig = plt.figure(figsize=(5 * cols, 5 * rows))

    folder_path = f'../plots/exploration/k_means/{name}/'
    os.makedirs(folder_path, exist_ok=True)

    for i, index in enumerate(indices):
        ax = fig.add_subplot(rows, cols, i + 1, projection='3d')
        plot_event(fig, events, index, ax)
        ax.set_title(f"Event: {index} | Label: {true_labels[i]}")        

    plt.tight_layout()
    plt.savefig(f'{folder_path}/matrix_plot.png')
    plt.close()

events = np.load("../../data_processing/O16/voxel_data/O16_size512.npy")
for i, cluster in enumerate(clusters):
    plot_events(cluster, f"cluster_{i}", events, true_labels[i])

**This section plots events of a particular region.**
- Change j to specify how many events to be plotted
- Change x1,x2,y1,y2 to set the boundaries of a region.

In [ ]:
def plot_selected_events(data, sampled_data, folder_path):
    fig = plt.figure()
    ax = fig.add_subplot(111) 

    j = 6
    i = random.randint(0, len(data) - 1) # corresponds to the random event
    maxim = 0
    while j != 0:  # number of events to be plotted
        # change the boundaries of a region
        x1 = 4
        y1 = 13
        x2 = 5
        y2 = 15

        if (maxim == len(data)): # solves an infinite for-loop problem
            break

        if (data[i][0] > x1 and data[i][0] < x2 and data[i][1] > y1 and data[i][1] < y2):
            fig = plt.figure(figsize=(10, 10))
            plot_event(fig, sampled_data, i)
            plt.savefig(f'{folder_path}/event_{i}.png')
            j-=1
        i = random.randint(0, len(data) - 1)
        maxim+=1

Applies **UMAP** embedding from ***clusering.py*** and saves the result in *../plots/exploration/umap*

In [ ]:
# apply UMAP embedding
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111)
label_names = ["0,1,2 Tracks", "3 Tracks", "4, 5 Tracks"]
neighbors = 150 # change or leave it

umap = UMAP_embedding(features, 2, labels, label_names, 0.5, neighbors)

In [ ]:
O16_experimental_data = np.load('../../data_processing/O16/voxel_data/O16_size512.npy')
experimental_data = O16_experimental_data[indices]

In [ ]:
#Plots events of a defined region. Change x1,x2,y1,y2 to set the boundaries of a region in the plot_selected_events section.
# plot_selected_events(data, sampled_data, folder_path)
#    data - clustered data 
#    sampled_data - point cloud data from which random point will be printed
#    folder_path - where files will be saved

name = '0,1,2 Tracks'
folder_path = f'../plots/exploration/umap/{name}/'
os.makedirs(folder_path, exist_ok=True)
plot_selected_events(umap_0, experimental_data, folder_path)

name = '3 Tracks'
folder_path = f'../plots/exploration/umap/{name}/'
os.makedirs(folder_path, exist_ok=True)
plot_selected_events(umap_1, experimental_data, folder_path)

name = '4, 5 Tracks'
folder_path = f'../plots/exploration/umap/{name}/'
os.makedirs(folder_path, exist_ok=True)
plot_selected_events(umap_2, experimental_data, folder_path)

Applies **T-SNE** embedding from ***clusering.py*** and saves the result in *../plots/exploration/t_sne*

In [ ]:
# apply T-SNE embedding
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111)
perplexity = 150 # change or leave it

tsne = t_SNE_clustering(features, 2, labels, label_names, 0.5, perplexity)

In [ ]:
O16_experimental_data = np.load('../../data_processing/O16/voxel_data/O16_size512.npy')
experimental_data = O16_experimental_data[indices]

In [ ]:
#Plots events of a defined region. Change x1,x2,y1,y2 to set the boundaries of a region in the plot_selected_events section.
# plot_selected_events(data, sampled_data, folder_path)
#    data - clustered data 
#    sampled_data - point cloud data from which random point will be printed
#    folder_path - where files will be saved

name = '0,1,2 Tracks'
folder_path = f'../plots/exploration/tsne/{name}/'
os.makedirs(folder_path, exist_ok=True)
plot_selected_events(tsne_0, experimental_data, folder_path)

name = '3 Tracks'
folder_path = f'../plots/exploration/tsne/{name}/'
os.makedirs(folder_path, exist_ok=True)
plot_selected_events(tsne_1, experimental_data, folder_path)

name = '4, 5 Tracks'
folder_path = f'../plots/exploration/tsne/{name}/'
os.makedirs(folder_path, exist_ok=True)
plot_selected_events(tsne_2, experimental_data, folder_path)